## Example 1: Package and kernel protocol versions

In [ ]:
import ipykernel

# Example 1: Checking kernel protocol version and package info
print("Example 1: Kernel protocol and version information")

# Get kernel protocol version
protocol_version = ipykernel.kernel_protocol_version
protocol_version_info = ipykernel.kernel_protocol_version_info
package_version_info = ipykernel.version_info

print(f"Kernel protocol information:")
print(f"  Protocol version: {protocol_version}")
print(f"  Protocol version tuple: {protocol_version_info}")
print(f"  Package version: {package_version_info}")

# Check version compatibility
print(f"\nVersion details:")
print(f"  Major protocol version: {protocol_version_info[0]}")
print(f"  Minor protocol version: {protocol_version_info[1]}")
print(f"  Package major version: {package_version_info[0]}")
print(f"  Package minor version: {package_version_info[1]}")
print(f"  Package micro version: {package_version_info[2]}")

# Format version strings
version_string = f"{package_version_info[0]}.{package_version_info[1]}.{package_version_info[2]}"
print(f"  Full package version: {version_string}")

import importlib.metadata
assert ipykernel.__version__ == importlib.metadata.version("ipykernel")
assert ipykernel.__version__.startswith(version_string)
assert protocol_version == ".".join(map(str, protocol_version_info))
assert protocol_version_info[0] == 5  # Jupyter messaging protocol 5.x

## Example 2: Comm objects

In [ ]:
from ipykernel.comm import Comm
import uuid

# Example 2: Creating and working with Comm objects
print("Example 2: Communication objects with Comm")

# Create communication objects with different target names
comm_targets = ['widget_comm', 'data_comm', 'control_comm']

print("Creating Comm objects:")
comms = []
for target in comm_targets:
    try:
        # Create comm with specific target name
        comm = Comm(target_name=target)
        comms.append(comm)
        
        print(f"  Target: {target}")
        print(f"    Comm ID: {comm.comm_id}")
        print(f"    Target name: {comm.target_name}")
        print(f"    Kernel attached: {comm.kernel is not None}")
        
    except Exception as e:
        print(f"  Target: {target} -> Error: {e}")
        raise

# Every comm gets its own id and keeps the target it was opened for
assert [c.target_name for c in comms] == comm_targets
assert len({c.comm_id for c in comms}) == len(comms)
assert all(len(c.comm_id) == 32 and int(c.comm_id, 16) >= 0 for c in comms)  # uuid4 hex

## Example 3: Kernel specs with get_kernel_dict and write_kernel_spec

In [ ]:
import json
import os
import tempfile

from ipykernel import kernelspec

# Example 3: Kernel specs with get_kernel_dict and write_kernel_spec
print("Example 3: Kernel specification information")

spec = kernelspec.get_kernel_dict()
print(f"  Kernel spec: {spec}")
assert spec["language"] == "python"
assert "argv" in spec and spec["argv"][1:4] == ["-m", "ipykernel_launcher", "-f"]
assert spec["argv"][-1] == "{connection_file}"
assert kernelspec.KERNEL_NAME.startswith("python")

# write_kernel_spec writes kernel.json (plus logos) into a directory without installing it
with tempfile.TemporaryDirectory() as temp_dir:
    path = kernelspec.write_kernel_spec(os.path.join(temp_dir, "spec"))
    files = sorted(os.listdir(path))
    print(f"  Files written: {files}")
    assert "kernel.json" in files
    with open(os.path.join(path, "kernel.json")) as f:
        written = json.load(f)
    assert written["argv"] == spec["argv"] and written["language"] == "python"

## Example 4: The kernel's caching compiler

In [ ]:
from ipykernel import compiler
import ast
import os

# Example 4: Code compilation with ipykernel compiler
print("Example 4: Code compilation functionality")

# Test code samples for compilation
code_samples = [
    'print("Hello from kernel")',
    'x = 42; y = x * 2',
    'def hello(): return "world"',
    'import os; print(os.name)',
    'result = [i**2 for i in range(5)]'
]

print("Testing code compilation:")

# Check compiler module attributes
compiler_attrs = [attr for attr in dir(compiler) if not attr.startswith('_')]
print(f"  Compiler module functions: {compiler_attrs[:5]}")

# Test compilation if compiler functions are available
for i, code in enumerate(code_samples, 1):
    try:
        print(f"  Code sample {i}: {code}")
        
        # Try to compile using standard Python compile
        compiled_code = compile(code, f'<sample_{i}>', 'exec')
        print(f"    Compilation: successful")
        print(f"    Bytecode type: {type(compiled_code).__name__}")
        
        # Parse AST for analysis
        ast_tree = ast.parse(code)
        print(f"    AST nodes: {len(ast_tree.body)}")
        
    except SyntaxError as e:
        print(f"    Compilation: syntax error - {e}")
    except Exception as e:
        print(f"    Compilation: error - {e}")
        raise

# murmur2_x86 is the hash the kernel uses to name cell files; it is deterministic
assert compiler.murmur2_x86("hello", 0) == 3848350155
assert compiler.murmur2_x86("hello", 0) != compiler.murmur2_x86("hello", 1)

# The kernel's compiler gives every cell a stable file name under its temp directory
caching_compiler = compiler.XCachingCompiler()
source = "x = 41 + 1"
filename = caching_compiler.get_code_name(source, None, 0)
assert filename == caching_compiler.get_code_name(source, None, 0)
assert os.path.dirname(filename) == compiler.get_tmp_directory()
namespace = {}
exec(caching_compiler(source, filename, "exec"), namespace)
assert namespace["x"] == 42

## Example 5: Connection files

In [ ]:
import json
import os
import tempfile

from IPython import get_ipython
from ipykernel import connect

# Example 5: Connection files
print("Example 5: Kernel connection utilities")

# write_connection_file writes the ports, key and transport a client needs to connect
with tempfile.TemporaryDirectory() as temp_dir:
    fname, info = connect.write_connection_file(os.path.join(temp_dir, "kernel-example.json"), key=b"secret")
    with open(fname) as f:
        on_disk = json.load(f)
    print(f"  Connection file keys: {sorted(on_disk)}")
    assert {"shell_port", "iopub_port", "stdin_port", "hb_port", "control_port", "ip", "key", "transport"} <= set(on_disk)
    assert on_disk["transport"] == "tcp" and on_disk["key"] == "secret"
    assert all(isinstance(on_disk[p], int) and on_disk[p] > 0 for p in ("shell_port", "iopub_port", "hb_port"))

# get_connection_file works only inside a running kernel
shell = get_ipython()
if shell is not None and getattr(shell, "kernel", None) is not None:
    path = connect.get_connection_file()
    print(f"  Running kernel's connection file: {os.path.basename(path)}")
    assert os.path.isfile(path)
    assert {"shell_port", "key", "transport"} <= set(json.loads(connect.get_connection_info()))
else:
    try:
        connect.get_connection_file()
    except RuntimeError as e:
        print(f"  Outside a kernel: {e}")
    else:
        raise AssertionError("get_connection_file() should fail outside a running kernel")